# Research source: Historical seasonal-results assessment

**Route:** archival methodology, not the default frozen-deployment quick start.  
**Original uploaded notebook:** `04_LP_Seasonal_Weekly_Results(4).ipynb`.  
**Purpose:** Processes the separate historical 3-Hz campaign. Its output is not a newly optimized LP35 result.

## Before executing
The public optimizer uses the supplied numerical includes and frozen coefficients. It **does not need this notebook to be rerun**. Start with `notebooks/00_Quick_Start.ipynb` and `GAMS_LaPalma/START_LAPALMA.gms` instead.

Full execution of this research notebook needs: Historical case, deployment, original native schedules and dynamic caches. These archival dependencies are not all in the lightweight public package. The exact original case/protocol bytes have not been fabricated. Original source-hash checks are intentionally strict and may reject comment-sanitized public copies; archival replay uses the exact private originals and the author's matching data archive.

## Governing model and units
UC dispatch and reserve are in MW; surviving inertia is in MW s; gain is in MW/Hz. The primary frequency criterion is a 46.5-Hz minimum at 50 Hz. The historical notebook branch remains 47 Hz. The SODE model uses one frequency state and two states per surviving governor; its response is limited above by individual headroom. The feature vector order is `loss_mw, inertia_mws, gain_mw_hz, reserve_mw`. A calibrated score becomes an affine GAMS inequality after inverse standardization and a probability-logit shift.

## Algorithm, interpretation and verification
Read the original Markdown and the stage annotations below in order. The detailed derivations and symbol-to-source mapping are in `docs/Mathematical_Formulation/Technical_Manual.pdf`. A grouping or calibration diagnostic is not a physical risk guarantee. Do not transfer a previous day's state through rounded Excel output; use native GDX.

**Curation status:** notebook structure, imports, Python syntax and exact preservation of all original code-cell text are checked. Full production execution is **NOT RUN**. Source-only functions may be exercised by the small verification notebooks using explicitly labelled fixtures; that does not validate the unavailable full experiment.

Original outputs and execution counts were removed from this public copy to avoid stale results and private workstation paths. The separately supplied private original backup preserves the original bytes.


# La Palma seasonal weekly results — evidence-preserving processor

This notebook processes the completed four-season La Palma campaign without rewriting the native optimization history. The original 0.5% MIP target is retained. Time-limit incumbents near 1% are included as descriptive daily evidence with their exact gaps, but missing downstream schedules are never invented. The analytical SP comparison is handled separately: the current source-derived 3-Hz spring-day workbook is evaluated as one day, while the submitted Raja/source 3.5-Hz comparator remains historical context rather than being mixed into the four-season 3-Hz table.


## Before starting
Wait until the native jobs and exports stop. This notebook never retrains or launches GAMS. Close Excel before writing the summary workbook. Complete weeks and failed/missing days remain separate; no six-day sum is called a weekly result.

The operational reader checks initial/final state identity, integer history, startup category, capacity, hourly ramps, aggregate balance, objective reconstruction and the actual affine cut. It does not claim a La Palma DC network check.

### Source code stage 3

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
print("KERNEL IS ALIVE")

### Source code stage 4

**Algorithm objects:** `find_project`, `sha_file`, `hash_object`, `atomic_json`, `atomic_csv`, `write_excel`, `excel_values`, `number`, `read_matrix`, `read_pairs`, `assert_sources`, `prepare_parents`, `event_inputs`, `event_key`, `integrate_event`, `evaluate_event`, `cache_open`, `cache_save`, `simulate_parent_library`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Production workload:** this stage may fit models or process a large event/campaign library. It was not run for packaging. Do not execute it merely to use the released frozen coefficients.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
# Shared definitions are embedded verbatim in the notebooks; no custom .py is needed.
from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED
from xml.etree import ElementTree as ET
import csv, hashlib, json, math, os, re, shutil, sys, time, tempfile
import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
from scipy.special import expit


def find_project():
    here = Path.cwd().resolve()
    for p in (here, here.parent):
        if (p/'Data_LaPalma/LP_Case.json').is_file() and (p/'GAMS_LaPalma').is_dir():
            return p
    raise FileNotFoundError('Open this notebook in LaPalma_SEGAN/Notebooks_LaPalma, not the IEEE project.')

PROJECT = find_project()
NB = PROJECT/'Notebooks_LaPalma'
DATA = PROJECT/'Data_LaPalma'
GAMS = PROJECT/'GAMS_LaPalma'
EVID = PROJECT/'Evidence'; EVID.mkdir(exist_ok=True)
CASE = json.loads((DATA/'LP_Case.json').read_text(encoding='utf-8'))
IDS = CASE['ids']; NGEN = len(IDS)
if IDS != [f'i{j}' for j in range(1, 12)]:
    raise ValueError('Expected the original ordered eleven-unit La Palma fleet.')
PMAX = np.array(CASE['pmax']); PMIN = np.array(CASE['pmin'])
HS = np.array(CASE['inertia_s']); MBASE = np.array(CASE['mbase'])
KG = np.array(CASE['gain_pu']); A1 = np.array(CASE['a1'])
A2 = np.array(CASE['a2']); B1 = np.array(CASE['b1'])
F0 = float(CASE['f0_hz']); FMIN = float(CASE['nadir_min_hz'])
if not (np.isclose(F0,50.) and np.isclose(F0-FMIN,3.)):
    raise ValueError('This SEGAN La Palma design requires nominal50 Hz and a 3-Hz drop (47-Hz threshold).')
EUNIT = HS*MBASE
GAIN = KG*MBASE/F0  # MW/Hz; source gain K*Mbase = F0*GAIN.
for vector in (PMAX, PMIN, HS, MBASE, KG, A1, A2, B1):
    if vector.shape != (11,) or not np.isfinite(vector).all():
        raise ValueError('Every unit-parameter vector must contain exactly eleven finite entries.')
CORE_FEATURES = ['loss_mw', 'inertia_mws', 'gain_mw_hz', 'reserve_mw']
NUMERIC = dict(method='LSODA', rtol=1e-8, atol=1e-10, max_step=.20,
               output_step=.02, duration=60., extension=120.,
               stop_hz=40., settle_range_hz=.005, settle_slope_hz_s=1e-4,
               nadir_refine_tol_hz=1e-3, boundary_hz=1e-4)
DYNAMIC_VERSION = 'LP_UNIT_ID_UPPER_CAP_HZ_SOURCE_DAMP_v3'
MODEL_SPEC = {k:CASE[k] for k in ['ids','pmax','inertia_s','mbase','gain_pu','a1','a2','b1',
                                      'f0_hz','damping_fraction_per_hz','nadir_min_hz',
                                      'rocof_max_hz_s','qss_drop_hz']}
MODEL_SPEC.update(version=DYNAMIC_VERSION, limiter='minimum(internal_response, unit_headroom)',
                  numeric=NUMERIC, units='Hz / MW / MW*s / MW-per-Hz')

def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda:handle.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

def hash_object(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,separators=(',',':'),allow_nan=False).encode()).hexdigest()

MODEL_ID = hash_object(MODEL_SPEC)[:16]
WORK = DATA/'Corrected'/MODEL_ID; WORK.mkdir(parents=True,exist_ok=True)

def atomic_json(path, value):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)),encoding='utf-8')
    os.replace(tmp,path)

def atomic_csv(frame,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp');frame.to_csv(tmp,index=False);os.replace(tmp,path)

def write_excel(path, sheets):
    """CSV/checkpoints are authoritative. Write/validate a NEW workbook, then replace.
    A Windows-locked old workbook is not deleted and never costs another simulation.
    All sheet names are keyword arguments (pandas 3-compatible).
    """
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.stem+'_writing_'+str(time.time_ns())+'.xlsx')
    try:
        with pd.ExcelWriter(tmp,engine='openpyxl') as writer:
            for name,frame in sheets.items():
                if not isinstance(frame,pd.DataFrame):frame=pd.DataFrame(frame)
                frame.to_excel(writer,sheet_name=name[:31],index=False)
                ws=writer.sheets[name[:31]];ws.freeze_panes='A2'
                ws.auto_filter.ref=ws.dimensions
                for c in ws[1]: c.font=c.font.copy(bold=True)
                for column_index in range(1,ws.max_column+1):
                    col=[ws.cell(row=row_index,column=column_index) for row_index in range(1,min(100,ws.max_row)+1)]
                    samples=[len(str(c.value)) for c in col if c.value is not None]
                    ws.column_dimensions[col[0].column_letter].width=min(38,max(12,max(samples,default=12)+2))
        with ZipFile(tmp) as z:
            if z.testzip() is not None or 'xl/workbook.xml' not in z.namelist():raise IOError('Invalid Excel export')
        try: os.replace(tmp,path)
        except PermissionError:
            path=path.with_name(path.stem+'_RECOVERED_'+str(time.time_ns())+'.xlsx');os.replace(tmp,path)
            print('The prior workbook is locked; saved a fresh workbook:',path)
    except Exception:
        print('Excel export failed. Saved CSVs/caches are retained; rerun only the output cell.')
        raise
    return path

# Reader uses stored cell values, including GAMS EPS, with no Excel automation.
def excel_values(path, sheet):
    ns={'x':'http://schemas.openxmlformats.org/spreadsheetml/2006/main'}
    rid='{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id'
    with ZipFile(path) as z:
        strings=[]
        if 'xl/sharedStrings.xml' in z.namelist():
            strings=[''.join(x.itertext()) for x in ET.fromstring(z.read('xl/sharedStrings.xml')).findall('x:si',ns)]
        rel={x.attrib['Id']:x.attrib['Target'] for x in ET.fromstring(z.read('xl/_rels/workbook.xml.rels'))}
        sheets=ET.fromstring(z.read('xl/workbook.xml')).find('x:sheets',ns)
        target=next((rel[s.attrib[rid]] for s in sheets if s.attrib['name']==sheet),None)
        if target is None:raise KeyError(f'{Path(path).name}: missing sheet {sheet}')
        target=target.lstrip('/') if target.startswith('/') else 'xl/'+target
        rows=[]
        for row in ET.fromstring(z.read(target)).findall('x:sheetData/x:row',ns):
            d={}
            for c in row:
                col=0
                for a in re.match(r'[A-Z]+',c.attrib['r']).group():col=col*26+ord(a)-64
                v=c.find('x:v',ns);typ=c.attrib.get('t')
                if typ=='s':value=strings[int(v.text)] if v is not None else None
                elif typ=='inlineStr':value=''.join(c.find('x:is',ns).itertext())
                elif v is None:value=None
                elif typ in ('str','e'):value=v.text
                else:
                    try:value=float(v.text)
                    except (ValueError,TypeError):value=v.text
                d[col-1]=value
            rows.append(d)
        width=max((max(r,default=-1)+1 for r in rows),default=0)
        return [[r.get(j) for j in range(width)] for r in rows]

def number(value):
    if value is None or value=='':return 0.
    if isinstance(value,str) and value.strip().lower() in ('eps','epsilon'):return 0.
    out=float(value)
    if not math.isfinite(out):raise ValueError(f'Nonfinite native number: {value}')
    return out

def read_matrix(path,sheet,row_ids,col_ids):
    data=excel_values(path,sheet);result=np.zeros((len(row_ids),len(col_ids)))
    headers=next((r for r in data if sum(str(x) in col_ids for x in r)>=1),None)
    if headers is None:raise ValueError(f'{sheet}: no expected column IDs')
    positions={str(x):j for j,x in enumerate(headers) if str(x) in col_ids}
    if set(positions)!=set(col_ids):raise ValueError(f'{sheet}: incomplete column identifiers')
    mapped={str(r[0]):r for r in data if r and str(r[0]) in row_ids}
    if set(mapped)!=set(row_ids):raise ValueError(f'{sheet}: incomplete row identifiers')
    for i,key in enumerate(row_ids):
        for j,col in enumerate(col_ids):result[i,j]=number(mapped[key][positions[col]])
    return result

def read_pairs(path,sheet,expected=None):
    result={}
    for row in excel_values(path,sheet):
        if len(row)>=2 and row[0] is not None and (expected is None or str(row[0]) in expected):
            try:result[str(row[0])]=number(row[1])
            except (ValueError,TypeError):continue
    return result

def assert_sources():
    for relative,expected in CASE['source_hashes'].items():
        actual=sha_file(PROJECT/relative)
        if actual!=expected:raise ValueError(f'Original source changed: {relative}; do not reuse this case identity.')


def prepare_parents(seed=20260920):
    """Reuse the historical pool. No new A-LHS generation or hidden i11 restriction."""
    path=DATA/'A-LHS_newDataretry.xlsx'
    power=np.asarray(excel_values(path,'generation'),float)
    demand=np.asarray(excel_values(path,'load'),float).reshape(-1)
    if power.shape[1]!=11 or len(demand)!=len(power):
        raise ValueError('The source already has eleven columns; never append a twelfth dummy unit.')
    if np.any(demand<16-1e-6) or np.any(demand>36+1e-6):raise ValueError('Historical parent-load domain differs from 16--36 MW')
    online=power>1e-8
    violations=(power<-1e-7).any(1)|(power>PMAX+1e-7).any(1)|((power<PMIN-1e-7)&online).any(1)
    violations|=np.abs(power.sum(1)-demand)>1e-6
    if violations.any():raise ValueError(f'{violations.sum()} source parents fail declared static screens; review, do not silently repair.')
    # A group is the complete pre-event dispatch (all its outages remain together).
    gid=[hashlib.sha256(np.round(p,6).astype('<f8').tobytes()).hexdigest()[:24] for p in power]
    frame=pd.DataFrame(power,columns=IDS);frame.insert(0,'parent_id',gid)
    frame.insert(0,'source_row',np.arange(1,len(power)+1));frame['demand_mw']=demand
    frame['original_filter']='historical A-LHS cost-screened source; selection bias retained'
    frame['static_screen_pass']=True
    # Equal dispatches belong to one group even if a future source has another demand.
    groups=sorted(set(gid));rng=np.random.default_rng(seed);order=np.array(groups,dtype=object)[rng.permutation(len(groups))]
    ends=np.round(np.cumsum([.60,.15,.10,.15])*len(groups)).astype(int)
    assignment={};start=0
    for role,end in zip(['development','calibration','selection','test'],ends):
        assignment.update({str(g):role for g in order[start:end]});start=end
    frame['role']=frame.parent_id.map(assignment)
    # Duplicate physically identical rows are not counted as independent examples.
    unique=frame.drop_duplicates(['parent_id','demand_mw']).copy().reset_index(drop=True)
    unique['demand_band']=np.minimum(39, np.floor((unique.demand_mw-16)/.5+1e-7).astype(int))
    blocks=np.asarray(CASE['blocks']);slopes=np.asarray(CASE['slopes_keur_per_mwh'])
    rem=unique[IDS].to_numpy().copy();cost=(rem>1e-8)@np.asarray(CASE['no_load_keur_h'])
    for j in range(3):q=np.minimum(rem,blocks[:,j]);cost+=(q*slopes[:,j]).sum(1);rem-=q
    unique['screen_cost_keur_h']=cost
    atomic_csv(frame,WORK/'Parent_source_map.csv');atomic_csv(unique,WORK/'Parents.csv')
    receipt=dict(source_rows=len(frame),unique_physical_parents=len(unique),unique_groups=len(groups),
        duplicate_rows=len(frame)-len(unique),candidate_outage_rows_source=int(online.sum()),
        unique_parent_outages=int((unique[IDS].to_numpy()>1e-8).sum()),unit11_online=int(online[:,10].sum()),
        role_counts=unique.groupby('role').size().to_dict(),source_sha256=sha_file(path),model_id=MODEL_ID,
        scope='Static generation/load/headroom/inertia inputs; no chronology, no LP network screening',
        demand_definition='Historical load equals total thermal output: treated as supplied reference damping load. Seasonal deployment uses actual gross demand. Not evidence of identical distributions.')
    atomic_json(WORK/'Parent_receipt.json',receipt)
    return unique,receipt


def event_inputs(power, demand_mw, outage_index, commitment=None):
    power=np.asarray(power,float).copy();demand_mw=float(demand_mw);j=int(outage_index)
    if power.shape!=(11,) or not np.isfinite(power).all() or demand_mw<0:raise ValueError('Invalid event input')
    on=(power>1e-8) if commitment is None else np.rint(np.asarray(commitment,float)).astype(bool)
    if on.shape!=(11,) or not on[j] or power[j]<=1e-8:raise ValueError('Outage must remove an online producing unit')
    if np.any(power<-1e-6) or np.any(power>PMAX*on+1e-6):raise ValueError('Invalid pre-event generator outputs')
    survive=on.copy();survive[j]=False
    idx=np.flatnonzero(survive);E=float(EUNIT[survive].sum());rho=np.maximum(PMAX-power,0)[idx]
    return dict(power=power,online=on,survive=survive,index=idx,E=E,rho=rho,
        gain=GAIN[idx],a1=A1[idx],a2=A2[idx],b1=B1[idx],loss=float(power[j]),
        damping=demand_mw*CASE['damping_fraction_per_hz'],demand=demand_mw,outage=j)


def event_key(power,demand_mw,outage_index,commitment=None):
    on=(np.asarray(power)>1e-8) if commitment is None else np.rint(commitment).astype(bool)
    identity=dict(model_id=MODEL_ID,power=np.round(power,9).tolist(),demand=round(float(demand_mw),9),
                  outage=IDS[int(outage_index)],commitment=on.astype(int).tolist())
    return hash_object(identity)


def integrate_event(power,demand_mw,outage_index,commitment=None,refined=False,return_trace=False):
    v=event_inputs(power,demand_mw,outage_index,commitment)
    if v['E']<=0:raise ValueError('No surviving inertia; no fictitious epsilon inertia is inserted')
    n=len(v['index']);rtol=NUMERIC['rtol']*(.1 if refined else 1.)
    atol=NUMERIC['atol']*(.1 if refined else 1.)
    step=NUMERIC['output_step']*(.5 if refined else 1.)
    mx=NUMERIC['max_step']*(.5 if refined else 1.)
    def response(y):return np.minimum(v['gain']*(y[1:1+n]+v['b1']*y[1+n:]),v['rho'])
    def rhs(t,y):
        df=y[0];q=y[1:1+n];dq=y[1+n:]
        dfdt=F0/(2*v['E'])*(response(y).sum()-v['loss']-v['damping']*df)
        ddq=(-q-v['a1']*dq-df)/v['a2']
        return np.concatenate(([dfdt],dq,ddq))
    def stop(t,y):return F0+y[0]-NUMERIC['stop_hz']
    stop.terminal=True;stop.direction=-1
    y0=np.zeros(1+2*n);duration=NUMERIC['duration'];pieces=[];start=0.;state=y0
    while True:
        sol=solve_ivp(rhs,(start,duration),state,method=NUMERIC['method'],rtol=rtol,atol=atol,
                      max_step=mx,dense_output=True,events=stop)
        if not sol.success or not np.isfinite(sol.y).all():raise RuntimeError(sol.message)
        end=float(sol.t[-1]);grid=np.linspace(start,end,max(2,int(np.ceil((end-start)/step))+1))
        Y=sol.sol(grid);frequency=F0+Y[0];pieces.append((grid,Y,sol))
        censored=bool(sol.t_events[0].size)
        tail=grid>=max(start,end-5.)
        slope=float(rhs(end,Y[:,-1])[0]);range_=float(np.ptp(frequency[tail]))
        settled=(not censored and end>=10 and range_<=NUMERIC['settle_range_hz'] and abs(slope)<=NUMERIC['settle_slope_hz_s'])
        if censored or settled or duration>=NUMERIC['extension']:break
        start=end;state=Y[:,-1];duration=NUMERIC['extension']
    times=np.concatenate([g if k==0 else g[1:] for k,(g,_,__) in enumerate(pieces)])
    states=np.concatenate([y if k==0 else y[:,1:] for k,(_,y,__) in enumerate(pieces)],axis=1)
    freq=F0+states[0];nadir=float(freq.min());tn=float(times[freq.argmin()])
    # Refine each sampled local minimum using the dense interpolation of its own interval.
    for grid,Y,sol in pieces:
        f=F0+Y[0]
        for k in np.flatnonzero((f[1:-1]<f[:-2])&(f[1:-1]<=f[2:]))+1:
            opt=minimize_scalar(lambda t:float(F0+sol.sol(t)[0]),bounds=(grid[k-1],grid[k+1]),method='bounded',options={'xatol':1e-9})
            if opt.fun<nadir:nadir=float(opt.fun);tn=float(opt.x)
    rtotal=np.array([response(y).sum() for y in states.T])
    record=dict(minimum_observed_hz=nadir,time_min_s=tn,endpoint_hz=float(freq[-1]),
        initial_rocof_hz_s=float(rhs(0,y0)[0]),settled=bool(settled),censored=bool(censored),
        simulated_seconds=float(times[-1]),tail_range_hz=range_,tail_slope_hz_s=slope,
        loss_mw=v['loss'],inertia_mws=v['E'],gain_mw_hz=float(v['gain'].sum()),
        reserve_mw=float(v['rho'].sum()),gain_source=float(F0*v['gain'].sum()),demand_mw=float(demand_mw))
    return record,dict(time_s=times,frequency_hz=freq,response_mw=rtotal) if return_trace else None


def evaluate_event(power,demand_mw,outage_index,commitment=None):
    key=event_key(power,demand_mw,outage_index,commitment);started=time.perf_counter()
    base=dict(event_key=key,model_id=MODEL_ID,outage_unit=IDS[int(outage_index)],outage_index=int(outage_index))
    try:
        coarse,_=integrate_event(power,demand_mw,outage_index,commitment)
        fine,_=integrate_event(power,demand_mw,outage_index,commitment,refined=True)
        delta=abs(coarse['minimum_observed_hz']-fine['minimum_observed_hz'])
        agree=(coarse['minimum_observed_hz']<FMIN)==(fine['minimum_observed_hz']<FMIN)
        numerical=bool(agree and delta<=NUMERIC['nadir_refine_tol_hz'] and coarse['censored']==fine['censored'])
        witnessed=bool(fine['minimum_observed_hz']<FMIN-NUMERIC['boundary_hz'])
        resolved=bool(numerical and fine['settled'] and not fine['censored'])
        if numerical and witnessed:status='UNSAFE_OBSERVED';label=-1
        elif resolved and fine['minimum_observed_hz']>=FMIN+NUMERIC['boundary_hz']:status='SAFE_CONDITIONAL';label=1
        else:status='REVIEW';label=None
        base.update(fine,status=status,y_safe=label,numerical_refinement_ok=numerical,nadir_refinement_abs_hz=delta,
            nadir_crossing_observed=witnessed,resolved_nadir=resolved,
            rocof_crossing=bool(abs(fine['initial_rocof_hz_s'])>CASE['rocof_max_hz_s']+1e-6),
            qss_unresolved=not fine['settled'],qss_crossing=bool(fine['settled'] and abs(F0-fine['endpoint_hz'])>CASE['qss_drop_hz']+1e-6),
            review_reason='' if status!='REVIEW' else ('numerical_refinement' if not numerical else 'unsettled_or_boundary'),
            error='')
    except Exception as exc:
        base.update(status='FAILED_RETAINED',y_safe=None,numerical_refinement_ok=False,nadir_crossing_observed=False,
                    resolved_nadir=False,qss_unresolved=True,error=repr(exc))
    base['calculation_seconds']=time.perf_counter()-started
    return base


def cache_open(path):
    path=Path(path);cache={}
    if path.exists():
        lines=path.read_text(encoding='utf-8').splitlines()
        for i,line in enumerate(lines):
            if not line.strip():continue
            try:row=json.loads(line)
            except json.JSONDecodeError:
                if i==len(lines)-1:
                    backup=path.with_suffix('.truncated_backup_'+str(time.time_ns()))
                    shutil.copyfile(path,backup)
                    path.write_text('\n'.join(lines[:i])+'\n',encoding='utf-8');break
                raise ValueError('Corrupt cache before its last line; preserve file for inspection')
            if row.get('model_id')!=MODEL_ID:raise ValueError('Cache physics/numerics identity differs; never reuse these labels')
            cache[row['event_key']]=row
    return cache

def cache_save(handle,row):
    handle.write(json.dumps(row,allow_nan=False,default=lambda x:x.item() if isinstance(x,np.generic) else str(x))+'\n');handle.flush()


def simulate_parent_library(parents, limit=None, n_jobs=2):
    """One cached physical calculation per unique dispatch/load/outage/model identity.
    Worker processes (not threads) isolate LSODA's integrator state. Results are
    appended before Excel/plotting. Interrupting a batch only loses unfinished tasks.
    """
    from joblib import Parallel, delayed
    chosen=parents if limit is None else parents.iloc[:int(limit)]
    cachepath=WORK/'Label_cache.jsonl';cached=cache_open(cachepath);rows=[];pending={};descriptors=[]
    for record in chosen.to_dict('records'):
        power=np.array([record[i] for i in IDS]);demand_mw=float(record['demand_mw'])
        for j in np.flatnonzero(power>1e-8):
            key=event_key(power,demand_mw,j)
            descriptors.append((key,record))
            if key not in cached:pending[key]=(power,demand_mw,int(j))
    jobs=list(pending.values());started=time.perf_counter();added=0
    print(len(descriptors),'event records;',len(jobs),'uncached integrations; worker processes:',n_jobs,flush=True)
    with cachepath.open('a',encoding='utf-8') as handle:
        if jobs:
            with Parallel(n_jobs=max(1,int(n_jobs)),backend='loky',batch_size=1) as pool:
                for start in range(0,len(jobs),32):
                    done=pool(delayed(evaluate_event)(p,d,j) for p,d,j in jobs[start:start+32])
                    for r in done:cache_save(handle,r);cached[r['event_key']]=r;added+=1
                    if added%128==0 or added==len(jobs):
                        print(f'{added}/{len(jobs)} newly calculated; {time.perf_counter()-started:.1f}s',flush=True)
    for key,record in descriptors:
        rows.append({**cached[key],'parent_id':record['parent_id'],'source_row':record['source_row'],
                     'role':record['role'],'demand_band':record['demand_band'],'screen_cost_keur_h':record['screen_cost_keur_h']})
    frame=pd.DataFrame(rows);atomic_csv(frame,WORK/'LP_SODE_Labels.csv')
    receipt=dict(model_id=MODEL_ID,parents=len(chosen),events=len(frame),new_events=added,
        status_counts=frame.status.value_counts().to_dict(),labels_sha256=sha_file(WORK/'LP_SODE_Labels.csv'),
        parent_sha256=sha_file(WORK/'Parents.csv'),production_complete=(len(chosen)==len(parents)),
        source_scope='fixed historical parent pool; all online single-generator events, no spatial extension')
    atomic_json(WORK/'Labels_receipt.json',receipt)
    return frame,receipt


### Source code stage 5

**Algorithm objects:** `read_cube`, `load_daily`, `parse_time_limit_incumbent`, `process_sp3hz_reference`, `process_seasons`, `publication_figures`, `collect_return`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

TIDS=[f't{k}' for k in range(1,25)]
COLORS=['#4B5563','#009E73','#0072B2','#56B4E9','#E69F00','#D55E00','#CC79A7','#756BB1','#17A2A4','#8C6D31','#BD5B7F','#83904B','#8F6F9E','#3B7A57']

# Post-hoc reporting policy approved after the 24-h native campaign.
# The ORIGINAL GAMS target remains 0.5%. We do not rewrite that history.
# Time-limit incumbents up to 1.0% are accepted for descriptive daily economics;
# 1.0--1.25% are retained as sensitivity-only evidence. No missing day is invented.
STRICT_GAP_LIMIT = 0.00500002
RELAXED_GAP_LIMIT = 0.0100005
RELAXED_SENSITIVITY_LIMIT = 0.0125
PRIMARY_RUNS_EXCLUDE = {'SP'}
SP3HZ_SOURCE = DATA/'DM_SP_spring_d2 raja3,0Hz.xlsx'

def read_cube(path,sheet,axis1,axis2,axis3):
    data=excel_values(path,sheet);header=next((r for r in data if sum(str(x) in axis3 for x in r)>=1),None)
    if header is None:raise ValueError(f'{sheet}: missing headers')
    pos={str(x):j for j,x in enumerate(header) if str(x) in axis3}
    if set(pos)!=set(axis3):raise ValueError(f'{sheet}: incomplete third-axis labels')
    out=np.zeros((len(axis1),len(axis2),len(axis3)));seen=set();last=None
    for row in data:
        if not row:continue
        if str(row[0]) in axis1:last=str(row[0])
        if len(row)<2 or str(row[1]) not in axis2 or last is None:continue
        key=(last,str(row[1]));seen.add(key)
        for k,c in enumerate(axis3):out[axis1.index(last),axis2.index(str(row[1])),k]=number(row[pos[c]])
    if len(seen)!=len(axis1)*len(axis2):raise ValueError(f'{sheet}: incomplete row identifiers ({len(seen)})')
    return out


def load_daily(path,run,season,day,dep,previous=None):
    if not Path(path).parent.joinpath('COMPLETE.txt').is_file():raise ValueError('Native day has no COMPLETE marker')
    identity={}
    for row in excel_values(path,'Identity'):
        if len(row)>1 and str(row[0]) in ('campaign','run','season','model'):identity[str(row[0])]=str(row[1])
    expected={'campaign':dep['campaign'],'run':run,'season':season,'model':MODEL_ID}
    if identity!=expected:raise ValueError(f'Wrong workbook identity: {identity}; expected {expected}')
    summ=read_pairs(path,'Summary')
    if summ.get('day')!=day or summ.get('horizon_hours')!=24 or summ.get('accepted')!=1:raise ValueError('Wrong day/horizon or unaccepted solve')
    if summ.get('modelstat') not in (1,8) or summ.get('solvestat') not in (1,3):raise ValueError('No accepted integer native schedule')
    # The original 0.5% target remains reported, but a targeted-recovery COMPLETE
    # schedule is not rejected solely because its recorded search gap is larger.
    initial=read_matrix(path,'Initial',IDS,['commitment','power_mw','on_age_h','off_age_h'])
    final=read_matrix(path,'Final',IDS,['commitment','power_mw','on_age_h','off_age_h'])
    declared=np.c_[np.asarray(CASE['initial_on_h'])>0,CASE['initial_power'],CASE['initial_on_h'],CASE['initial_off_h']]
    expectedinitial=declared if day==1 else previous
    if expectedinitial is None or not np.allclose(initial,expectedinitial,atol=1e-6,rtol=0):raise ValueError('Initial state/previous-day transfer differs')
    power=read_matrix(path,'Dispatch',TIDS,IDS);u0=read_matrix(path,'Commitment',TIDS,IDS)
    starts=read_matrix(path,'Startup',TIDS,IDS);stops=read_matrix(path,'Shutdown',TIDS,IDS)
    onraw=read_matrix(path,'OnAge',TIDS,IDS);offraw=read_matrix(path,'OffAge',TIDS,IDS)
    b=read_cube(path,'Blocks',TIDS,IDS,['b1','b2','b3'])
    sc=read_cube(path,'StartCategories',TIDS,IDS,[f'j{k}' for k in range(1,9)])
    curtp=read_pairs(path,'Curtailment',TIDS);dp=read_pairs(path,'Demand',TIDS);wp=read_pairs(path,'WindAvailable',TIDS)
    if any(len(x)!=24 for x in [curtp,dp,wp]):raise ValueError('Missing hourly demand/wind/curtailment values')
    demand=np.array([dp[t] for t in TIDS]);wind=np.array([wp[t] for t in TIDS]);curt=np.array([curtp[t] for t in TIDS])
    if not np.allclose(demand,CASE['demand'][season][day-1],atol=1e-7,rtol=0) or not np.allclose(wind,CASE['res'][season][day-1],atol=1e-7,rtol=0):raise ValueError('Season/day input profile changed')
    binaryerr=max(float(np.max(abs(x-np.rint(x)))) for x in [u0,starts,stops,sc])
    if binaryerr>1e-5:raise ValueError('Nonbinary commitment/switch/category')
    u=np.rint(u0);sy=np.rint(starts);sz=np.rint(stops);sc=np.rint(sc)
    ageon=[];ageoff=[];on=np.rint(initial[:,2]);off=np.rint(initial[:,3]);uprev=np.rint(initial[:,0]);pprev=initial[:,1]
    violations=[]
    def check(name,err,tol=2e-5):
        err=float(np.max(np.asarray(err)))
        if err>tol:violations.append((name,err,tol))
        return err
    for k in range(24):
        check('binary transition',abs(u[k]-uprev-sy[k]+sz[k]),1e-6)
        check('exclusive switching',sy[k]+sz[k]-1,1e-6)
        check('minimum up obligation on shutdown',np.where(sz[k]>0,np.array(CASE['min_up_h'])-on,0),1e-6)
        check('minimum down obligation on startup',np.where(sy[k]>0,np.array(CASE['min_down_h'])-off,0),1e-6)
        thresholds=np.asarray(CASE['start_lower_h']);cat=(thresholds<=off[:,None]+1e-8).sum(1)-1
        expectedcat=np.zeros((11,8))
        for i in range(11):
            if sy[k,i]>.5:
                if cat[i]<0:violations.append(('no eligible startup category',i,0))
                else:expectedcat[i,cat[i]]=1
        check('startup category',abs(sc[k]-expectedcat),1e-6)
        check('ramp up',power[k]-pprev-np.array(CASE['ramp_up_mw_h']))
        check('ramp down',pprev-power[k]-np.array(CASE['ramp_down_mw_h']))
        on=(on+1)*u[k];off=(off+1)*(1-u[k]);ageon.append(on.copy());ageoff.append(off.copy())
        uprev=u[k];pprev=power[k]
    ageon=np.array(ageon);ageoff=np.array(ageoff)
    check('raw on-age drift',abs(onraw-ageon),1e-3);check('raw off-age drift',abs(offraw-ageoff),1e-3)
    check('rounded on-age agreement',abs(np.rint(onraw)-ageon),0)
    check('rounded off-age agreement',abs(np.rint(offraw)-ageoff),0)
    check('final state',abs(final-np.c_[u[-1],power[-1],ageon[-1],ageoff[-1]]),1e-6)
    check('generation lower',PMIN*u-power);check('generation upper',power-PMAX*u);check('negative generation',-power)
    check('block upper',b-np.asarray(CASE['blocks'])[None,:,:]*u[:,:,None]);check('negative block',-b)
    check('block sum',abs(b.sum(2)-power));balance=check('island power balance',abs(power.sum(1)+wind-curt-demand))
    check('negative curtailment',-curt);check('excess curtailment',curt-wind)
    method=next(r for r in dep['runs'] if r['run']==run);frows=[];native_margin=read_matrix(path,'CutMargin',TIDS,IDS)

    sp_extra={}
    if method['mode']==3:
        knots1=np.array([14.,26.,38.,52.]);knots2=np.array([5.,11.,17.,22.]);knotsp=np.array([0.,2.5,5.,7.5,10.,12.5])
        w1=read_cube(path,'SPLambda1',TIDS,IDS,[f'x1r{n}' for n in range(1,5)])
        w2=read_cube(path,'SPLambda2',TIDS,IDS,[f'x2r{n}' for n in range(1,5)])
        wp=read_cube(path,'SPLambdaP',TIDS,IDS,[f'pr{n}' for n in range(1,7)])
        sel1=read_cube(path,'SPSelectors1',TIDS,IDS,[f'x1r{n}' for n in range(1,5)])
        sel2=read_cube(path,'SPSelectors2',TIDS,IDS,[f'x2r{n}' for n in range(1,5)])
        selp=read_cube(path,'SPSelectorsP',TIDS,IDS,[f'pr{n}' for n in range(1,7)])
        for w,sel,target,name in [(w1,sel1,np.ones_like(u),'x1'),(w2,sel2,np.ones_like(u),'x2'),(wp,selp,u,'loss')]:
            check('SP negative weights '+name,-w)
            check('SP weight sum '+name,abs(w.sum(2)-target))
            check('SP binary selectors '+name,abs(sel-np.rint(sel)),1e-5)
            check('SP selected interval count '+name,abs(sel[:,:,1:].sum(2)-target))
            # Preserve the exact source adjacent-interval construction, not the IEEE selector variant.
            allowed=np.zeros_like(w);allowed[:,:,1:]+=sel[:,:,1:];allowed[:,:,:-1]+=sel[:,:,1:]
            check('SP weight/interval linkage '+name,w-allowed)
        z1=w1@knots1;z2=w2@knots2
        Eall=(u@EUNIT)[:,None]-u*EUNIT
        Rall=((PMAX*u-power).sum(1))[:,None]-(PMAX*u-power)
        check('SP inertia coordinate',abs((z1+z2)/.5-Eall))
        check('SP reserve coordinate',abs((z1-z2)/2.-Rall))
        check('SP loss coordinate',abs(wp@knotsp-power))
        check('SP x1 lower',14.-z1);check('SP x1 upper',z1-52.)
        check('SP x2 lower',5.-z2);check('SP x2 upper',z2-21.)
        proxy=(w1@(knots1**2)-w2@(knots2**2))/(.5*2.)
        p2=wp@(knotsp**2);credit=.01*demand[:,None]*6.*power*50.*.25
        proxy_margin=proxy-(50.*6./(4.*3.))*p2+credit
        parent_margin=Eall*Rall-(50.*6./(4.*3.))*power**2+credit
        check('SP enforced proxy',-proxy_margin,1e-4)
        # A parent violation is a reported approximation difference, NOT a reason to discard a source-feasible SP schedule.
        sp_extra={'sp_proxy_product_mw2s':proxy,'sp_parent_product_mw2s':Eall*Rall,
                  'sp_loss_squared_proxy':p2,'sp_proxy_margin_mw2s':proxy_margin,
                  'sp_parent_margin_mw2s':parent_margin,
                  'sp_parent_violation':parent_margin < -1e-4}
    for k in range(24):
        for j in range(11):
            survivor=u[k].copy();survivor[j]=0
            E=float(EUNIT@survivor);G=float(GAIN@survivor);R=float(((PMAX*u[k]-power[k])* (np.arange(11)!=j)).sum());loss=float(power[k,j])
            if method['mode']>0:
                check('RoCoF cut',loss-2*E*CASE['rocof_max_hz_s']/F0)
                check('headroom response condition',loss-R)
            margin=None
            if method['mode']==2:
                c=dep['cuts'][run];margin=c['k0']+c['kL']*loss+c['kE']*E+c['kG']*G+c['kR']*R+c['bigM_by_outage'][j]*(1-u[k,j])
                check('affine nadir cut',-margin,1e-5);check('exported cut margin',abs(native_margin[k,j]-margin),1e-5)
            frows.append(dict(run=run,season=season,day=day,hour=k+1,global_hour=(day-1)*24+k+1,outage_unit=IDS[j],candidate_online=int(u[k,j]),loss_mw=loss,inertia_mws=E,gain_mw_hz=G,reserve_mw=R,cut_margin=margin,**{name:float(arr[k,j]) for name,arr in sp_extra.items()}))
    costs=dict(production_keur=float((b*np.asarray(CASE['slopes_keur_per_mwh'])[None,:,:]).sum()),fixed_keur=float((u@np.asarray(CASE['no_load_keur_h'])).sum()),startup_keur=float((sc*np.asarray(CASE['start_cost_keur'])[None,:,:]).sum()),curtailment_cost_keur=0.)
    for key,val in costs.items():check(key,abs(summ[key]-val),1e-5)
    check('objective reconstruction',abs(summ['objective_keur']-sum(costs.values())),1e-5)
    check('EUR conversion',abs(summ['objective_eur']-1000*summ['objective_keur']),1e-3)
    if violations:raise ValueError(f'{run}/{season}/d{day}: {violations[:8]}')
    return dict(summary=summ,power=power,commitment=u,demand=demand,wind=wind,curtailment=curt,initial=initial,final=final,features=frows,balance_residual_mw=balance)


def parse_time_limit_incumbent(folder, run, season, day):
    """Read the native solver log only. This never fabricates the missing schedule.

    A time-limit incumbent can support objective/bound/gap/runtime reporting, but
    without an exported workbook/state it cannot support post-UC ODE evaluation,
    state handoff, or a seven-day total. The original 0.5% target remains recorded.
    """
    folder=Path(folder); log=folder/'LaPalma_FCUC.log'
    if not log.is_file():
        return dict(run=run,season=season,day=day,status='FAILED_SOLVE_NO_LOG',native_evidence='log missing')
    text=log.read_text(errors='ignore')
    no_integer=('no integer solution' in text.lower()) or ('No solution exists' in text)
    def last_float(pattern):
        vals=re.findall(pattern,text,re.I|re.M)
        return float(vals[-1]) if vals else np.nan
    obj=last_float(r'^\s*MIP Solution:\s*([+\-0-9.Ee]+)')
    bound=last_float(r'^\s*Best possible:\s*([+\-0-9.Ee]+)')
    gap=last_float(r'^\s*Relative gap:\s*([+\-0-9.Ee]+)')
    # Child-job elapsed wall time is the best end-to-end daily timing record.
    elapsed=re.findall(r'elapsed\s+(\d+):(\d+):(\d+(?:\.\d+)?)',text,re.I)
    wall=np.nan
    if elapsed:
        h,m,s=elapsed[-1];wall=3600*float(h)+60*float(m)+float(s)
    fixed_infeasible=('Fixed MIP status(3): infeasible' in text) or ('Final solve did not return an optimal solution' in text)
    if no_integer or not np.isfinite(obj):
        tier='NO_INTEGER_INCUMBENT'
    elif fixed_infeasible:
        tier='RELAXED_INCUMBENT_NUMERICAL_REVIEW'
    elif gap <= RELAXED_GAP_LIMIT:
        tier='RELAXED_INCUMBENT_LE_1PCT'
    elif gap <= RELAXED_SENSITIVITY_LIMIT:
        tier='RELAXED_INCUMBENT_1_TO_1P25PCT'
    else:
        tier='TIME_LIMIT_INCUMBENT_OUTSIDE_RELAXED_POLICY'
    return dict(run=run,season=season,day=day,status=tier,
                objective_keur=obj,objective_eur=1000*obj if np.isfinite(obj) else np.nan,
                best_bound_keur=bound,best_bound_eur=1000*bound if np.isfinite(bound) else np.nan,
                relative_gap=gap,wall_seconds=wall,resource_seconds=1200.,
                fixed_incumbent_numerical_review=bool(fixed_infeasible),
                native_evidence='LaPalma_FCUC.log + failed.gdx summary only; no exported schedule/state')


def process_sp3hz_reference(out):
    """Evaluate the dedicated 3-Hz SP source workbook as ONE DAY only.

    This is not promoted to a four-season/week result. The historic 3.5-Hz SP
    comparator from the submitted study remains contextual evidence and is not
    pooled with this 3-Hz day.
    """
    summary=[];events=[]
    if not SP3HZ_SOURCE.is_file():
        summary.append(dict(source=str(SP3HZ_SOURCE),status='SOURCE_MISSING',criterion_hz=47.0))
        return pd.DataFrame(summary),pd.DataFrame(events)
    raw=pd.read_excel(SP3HZ_SOURCE,sheet_name='gen',index_col=0)
    raw=raw.reindex(index=TIDS,columns=IDS)
    power=raw.apply(pd.to_numeric,errors='coerce').fillna(0.).to_numpy(float)
    if power.shape!=(24,11) or not np.isfinite(power).all():
        raise ValueError('SP 3-Hz source generation table is not a finite 24x11 schedule')
    commitment=(power>1e-8).astype(int)
    demand=np.asarray(CASE['demand']['spring'][1],float)  # file name explicitly identifies spring d2
    hourly=[]
    for k in range(24):
        hour_events=[]
        for j in np.flatnonzero(commitment[k]>.5):
            e=evaluate_event(power[k],demand[k],int(j),commitment[k])
            row={**e,'run':'SP_3HZ_ONE_DAY','season':'spring','day':2,'hour':k+1,'global_hour':k+1,'outage_unit':IDS[j]}
            events.append(row);hour_events.append(row)
        if hour_events:
            hourly.append(min(float(r['minimum_observed_hz']) for r in hour_events))
        else: hourly.append(np.nan)
    E=pd.DataFrame(events)
    def cnt(c):return int(E[c].fillna(False).sum()) if c in E else 0
    summary.append(dict(source=SP3HZ_SOURCE.name,status='ONE_DAY_3HZ_DYNAMIC_COMPARATOR',season='spring',day=2,
                        criterion_hz=47.0,hours=24,events=len(E),nadir_crossings=cnt('nadir_crossing_observed'),
                        rocof_crossings=cnt('rocof_crossing'),qss_crossings=cnt('qss_crossing'),
                        censored=cnt('censored'),resolved_nadirs=cnt('resolved_nadir'),
                        review_events=int(E.status.isin(['REVIEW','FAILED_RETAINED']).sum()) if 'status' in E else 0,
                        min_observed_hz=float(E.minimum_observed_hz.min()) if len(E) else np.nan,
                        note='Current 3-Hz source-day comparator only; not a seven-day seasonal SP result. Historic 3.5-Hz SP remains legacy context.'))
    pd.DataFrame({'hour':np.arange(1,25),'worst_observed_hz':hourly}).to_csv(out/'SP_3Hz_hourly_worst.csv',index=False)
    return pd.DataFrame(summary),E


def process_seasons(dep):
    root=GAMS/'Results_Seasonal'/dep['campaign'];out=root/'Processed';out.mkdir(parents=True,exist_ok=True)
    cachepath=WORK/'Schedule_event_cache.jsonl';cached=cache_open(cachepath)
    days=[];hours=[];events=[];features=[];incumbents=[]
    primary_modes=[m for m in dep['runs'] if m['run'] not in PRIMARY_RUNS_EXCLUDE]
    with cachepath.open('a',encoding='utf-8') as handle:
        for season in CASE['seasons']:
            for mode in primary_modes:
                run=mode['run'];prev=None;broken=False
                for day in range(1,8):
                    folder=root/run/season/f'd{day}';book=folder/f'Results_{run}_{season}_d{day}.xlsx'
                    if broken:
                        days.append(dict(run=run,season=season,day=day,status='BLOCKED_PREDECESSOR',native_evidence='no solve because prior day lacked accepted state export'))
                        continue
                    if (folder/'COMPLETE.txt').exists():
                        try:rec=load_daily(book,run,season,day,dep,prev)
                        except Exception as exc:
                            days.append(dict(run=run,season=season,day=day,status='READER_REVIEW',error=repr(exc)));broken=True;continue
                        prev=rec['final']
                        recovered=(folder/'RECOVERY_POLICY.txt').is_file()
                        day_status='OPERATIONAL_CHECKS_PASSED_RECOVERED' if recovered else 'OPERATIONAL_CHECKS_PASSED_STRICT'
                        days.append({**rec['summary'],'run':run,'season':season,'day':day,'status':day_status,'balance_residual_mw':rec['balance_residual_mw'],
                                     'gap_policy':'feasible targeted-recovery incumbent; actual gap reported' if recovered else 'original <=0.5% native target',
                                     'recovery_schedule':bool(recovered)})
                        features+=rec['features']
                        for k in range(24):
                            gh=(day-1)*24+k+1;power=rec['power'][k];commitment=rec['commitment'][k]
                            hours.append(dict(run=run,season=season,global_hour=gh,thermal_mw=float(power.sum()),demand_mw=rec['demand'][k],wind_available_mw=rec['wind'][k],wind_served_mw=rec['wind'][k]-rec['curtailment'][k],curtailed_mw=rec['curtailment'][k],online_units=int(commitment.sum()),unit11_online=int(commitment[10])))
                            for j in np.flatnonzero(commitment>.5):
                                key=event_key(power,rec['demand'][k],j,commitment)
                                if key not in cached:
                                    e=evaluate_event(power,rec['demand'][k],j,commitment);cache_save(handle,e);cached[key]=e
                                events.append(dict(**cached[key],run=run,season=season,day=day,global_hour=gh,unit11_outside_training=bool(commitment[10]>.5)))
                    elif (folder/'FAILED_SOLVE.txt').exists():
                        info=parse_time_limit_incumbent(folder,run,season,day);incumbents.append(info);days.append(info);broken=True
                    else:
                        days.append(dict(run=run,season=season,day=day,status='MISSING_NATIVE_DAY',native_evidence='no COMPLETE or FAILED_SOLVE marker'));broken=True
                    atomic_csv(pd.DataFrame(events),out/'All_events.csv');atomic_csv(pd.DataFrame(days),out/'Daily_status.csv')
                    print(season,run,'day',day,'processed;',len(events),'event records',flush=True)
    D=pd.DataFrame(days);H=pd.DataFrame(hours);E=pd.DataFrame(events);F=pd.DataFrame(features);I=pd.DataFrame(incumbents);weekly=[]
    for season in CASE['seasons']:
        for mode in primary_modes:
            run=mode['run'];r=D[(D.season==season)&(D.run==run)]
            strict=r[r.status=='OPERATIONAL_CHECKS_PASSED_STRICT']
            recovered=r[r.status=='OPERATIONAL_CHECKS_PASSED_RECOVERED']
            operational=pd.concat([strict,recovered],ignore_index=True,sort=False)
            rel=r[r.status.isin(['RELAXED_INCUMBENT_LE_1PCT','RELAXED_INCUMBENT_1_TO_1P25PCT'])];review_inc=r[r.status=='RELAXED_INCUMBENT_NUMERICAL_REVIEW']
            complete=len(operational)==7
            event=E[(E.season==season)&(E.run==run)] if not E.empty else E
            row=dict(run=run,season=season,status='COMPLETE_WEEK' if complete else 'PARTIAL_WEEK',
                     strict_schedule_days=len(strict),recovered_schedule_days=len(recovered),relaxed_summary_days=len(rel),numerical_review_incumbent_days=len(review_inc),reported_days=len(operational)+len(rel)+len(review_inc),events=len(event),
                     probability_floor=mode['pstar'],
                     evidence_note='Relaxed time-limit incumbents contribute daily objective/bound/gap only; no schedule, state handoff, or frequency event is invented.')
            reported=pd.concat([strict,rel],ignore_index=True,sort=False)
            if len(reported):
                row['reported_period_cost_eur']=float(reported.objective_eur.dropna().sum())
                row['reported_period_days']=int(reported.objective_eur.notna().sum())
                row['max_reported_gap']=float(reported.relative_gap.dropna().max()) if reported.relative_gap.notna().any() else np.nan
            if complete:
                for col in ['objective_eur','production_keur','fixed_keur','startup_keur','curtailment_cost_keur','wall_seconds','resource_seconds','curtailment_mwh','wind_served_mwh','demand_mwh']:
                    row[col]=float(operational[col].sum())
                row.update(mean_daily_cost_eur=float(operational.objective_eur.mean()),
                           mean_daily_gap=float(operational.relative_gap.mean()),
                           max_daily_gap=float(operational.relative_gap.max()),
                           recovered_schedule_days=int(len(recovered)),
                           week_contains_recovery=bool(len(recovered)>0))
            if not event.empty:
                def cnt(c):return int(event[c].fillna(False).sum()) if c in event else 0
                row.update(nadir_crossings=cnt('nadir_crossing_observed'),rocof_crossings=cnt('rocof_crossing'),qss_crossings=cnt('qss_crossing'),qss_unresolved=cnt('qss_unresolved'),censored=cnt('censored'),resolved_nadirs=cnt('resolved_nadir'),review_events=int(event.status.isin(['REVIEW','FAILED_RETAINED']).sum()),unit11_out_of_domain_events=cnt('unit11_outside_training'),min_observed_hz=float(event.minimum_observed_hz.min()))
            weekly.append(row)
    W=pd.DataFrame(weekly)
    across=[]
    for mode in primary_modes:
        run=mode['run'];q=W[W.run==run]
        full=q[q.status=='COMPLETE_WEEK']
        across.append(dict(run=run,complete_seasons=len(full),partial_seasons=int((q.status!='COMPLETE_WEEK').sum()),
                           mean_weekly_cost_eur=float(full.objective_eur.mean()) if len(full) else np.nan,
                           total_complete_week_cost_eur=float(full.objective_eur.sum()) if len(full) else np.nan,
                           max_daily_gap_complete=float(full.max_daily_gap.max()) if len(full) else np.nan,
                           note='Cross-season averages use complete seven-day histories only.'))
    SPsum,SPevents=process_sp3hz_reference(out)
    sp_campaign=[]
    # Record the attempted native SP day-1 solves separately; these are NOT part of the primary 392-job production denominator.
    for season in CASE['seasons']:
        folder=root/'SP'/season/'d1'
        if (folder/'FAILED_SOLVE.txt').exists(): sp_campaign.append(parse_time_limit_incumbent(folder,'SP',season,1))
    tables={'Seasonal_summary':W,'Across_seasons':pd.DataFrame(across),'Daily_status':D,'Relaxed_incumbents':I,
            'Hourly_dispatch':H,'All_events':E,'Features_and_margins':F,
            'SP_3Hz_one_day_summary':SPsum,'SP_3Hz_one_day_events':SPevents,
            'SP_native_day1_attempts':pd.DataFrame(sp_campaign),
            'Gap_policy':pd.DataFrame([
                dict(policy='Original native target',relative_gap_limit=0.005,role='Strict seven-day comparability'),
                dict(policy='User-approved relaxed daily reporting',relative_gap_limit=0.0100005,role='Descriptive objective/bound/gap for time-limit incumbent; not schedule acceptance'),
                dict(policy='Sensitivity-only retention',relative_gap_limit=0.0125,role='Keep near-1% incumbent visible; not primary relaxed tier'),
                dict(policy='SP current comparator',relative_gap_limit=np.nan,role='Dedicated spring d2 source workbook at 3-Hz criterion; one day only'),
                dict(policy='SP historical comparator',relative_gap_limit=np.nan,role='Original submitted Raja/source SP at 3.5-Hz criterion; contextual, not pooled with current 3-Hz results')])}
    for name,frame in tables.items():atomic_csv(frame,out/(name+'.csv'))
    atomic_json(out/'Processing_receipt.json',dict(campaign=dep['campaign'],model_id=MODEL_ID,
                primary_planned_days=len(primary_modes)*4*7,
                strict_schedule_days=int((D.status=='OPERATIONAL_CHECKS_PASSED_STRICT').sum()),
                recovered_schedule_days=int((D.status=='OPERATIONAL_CHECKS_PASSED_RECOVERED').sum()),
                relaxed_incumbent_days=int(D.status.isin(['RELAXED_INCUMBENT_LE_1PCT','RELAXED_INCUMBENT_1_TO_1P25PCT']).sum()),
                numerical_review_incumbent_days=int((D.status=='RELAXED_INCUMBENT_NUMERICAL_REVIEW').sum()),
                event_records=len(E),sp_scope='one-day 3-Hz current comparator; historical 3.5-Hz context kept separate',
                scope='Per-season summaries; complete-week economics use seven exported schedules only.'))
    return out,tables

def publication_figures(out,tables):
    out=Path(out);figs=out/'Figures';figs.mkdir(exist_ok=True);register=[]
    W=tables['Seasonal_summary'];E=tables['All_events'];H=tables['Hourly_dispatch'];I=tables['Relaxed_incumbents']
    def finish(fig,name,caption):
        fig.tight_layout();fig.savefig(figs/(name+'.pdf'),bbox_inches='tight');fig.savefig(figs/(name+'.png'),dpi=600,bbox_inches='tight');plt.close(fig);register.append(dict(figure=name,caption=caption))
    for season in CASE['seasons']:
        w=W[(W.season==season)&(W.status=='COMPLETE_WEEK')]
        if not w.empty:
            fig,ax=plt.subplots(figsize=(11,5));ix=np.arange(len(w));cols=[COLORS[k%len(COLORS)] for k in ix]
            ax.bar(ix,w.objective_eur/1000,color=cols,width=.68);ax.set_xticks(ix,w.run,rotation=45,ha='right');ax.set_ylabel('Weekly cost (kEUR)');ax.set_title(season.title()+' -- complete seven-day histories only')
            ay=ax.twinx();ay.plot(ix,w.wall_seconds,'D--',color='#263850',label='Sum of daily solver wall times');ay.set_yscale('log');ay.set_ylabel('Solver wall time (seconds; logarithmic)');ay.legend(loc='upper left',bbox_to_anchor=(0,1.13),frameon=False)
            finish(fig,'Cost_Runtime_'+season,'Costs in EUR. Only complete seven-day histories are compared; relaxed time-limit incumbents are not extrapolated into missing day 7.')
        ev=E[(E.season==season)&(E.resolved_nadir.fillna(False))] if not E.empty else E
        for density in [False,True]:
            fig,ax=plt.subplots(figsize=(8,4.5));drawn=0
            for k,run in enumerate(['BC','RESERVE','LR_C1','PA']):
                q=ev[ev.run==run].minimum_observed_hz.dropna().to_numpy() if not ev.empty else np.array([])
                total=int(((E.season==season)&(E.run==run)).sum()) if not E.empty else 0
                if len(q)<2:continue
                if density:
                    if np.std(q)<1e-8:continue
                    grid=np.linspace(min(40,float(q.min())-.2),50.2,800);ax.plot(grid,gaussian_kde(q)(grid),color=COLORS[k],label=f'{run}: {len(q)}/{total}')
                else:ax.hist(q,bins=np.linspace(min(40,float(q.min())-.1),50,51),histtype='step',density=True,color=COLORS[k],label=f'{run}: {len(q)}/{total}')
                drawn+=1
            if drawn:
                ax.axvline(FMIN,ls='--',color='#555555');ax.set_xlabel('Settled, uncensored, numerically accepted nadir (Hz)');ax.set_ylabel('Conditional density');ax.legend();ax.set_title(season.title()+' -- resolved subset; exclusions remain explicit')
                finish(fig,('KDE_' if density else 'Histogram_')+season,'Only resolved nadirs are shown. Censored, failed and unsettled events are not safe observations.')
            else:plt.close(fig)
        if not w.empty:
            fig,ax=plt.subplots(figsize=(9,4.5));r=w[['run','nadir_crossings','review_events']];pos=np.arange(len(r));ax.bar(pos-.18,r.nadir_crossings,width=.36,label='Witnessed nadir crossings',color='#BB5439');ax.bar(pos+.18,r.review_events,width=.36,label='Review / failed',color='#9AA3AA');ax.set_xticks(pos,r.run,rotation=45,ha='right');ax.set_ylabel('Event records (classes may overlap)');ax.legend();ax.set_title(season.title()+' -- frequency outcome accounting');finish(fig,'Events_'+season,'The denominator is online-generator events, not 168; review outcomes remain explicit.')
        for run in ['LR_C1','PA']:
            hh=H[(H.season==season)&(H.run==run)]
            if len(hh)!=168:continue
            fig,ax=plt.subplots(figsize=(11,4.5));ax.bar(hh.global_hour,hh.thermal_mw,color='#CC8052',label='Thermal');ax.bar(hh.global_hour,hh.wind_served_mw,bottom=hh.thermal_mw,color='#269988',label='RES served');ax.plot(hh.global_hour,hh.demand_mw,color='#223A5E',lw=1,label='Demand');ax.set_xlabel('Hour of representative week');ax.set_ylabel('MW');ax.legend(ncol=3,frameon=False);ax.set_title(f'{season.title()} -- {run} dispatch');finish(fig,'Dispatch_'+season+'_'+run,'Aggregate island generation; no nodal-network claim is made for La Palma.')
    if not I.empty:
        q=I[np.isfinite(I.relative_gap)]
        if len(q):
            fig,ax=plt.subplots(figsize=(10,4.8));labels=[f"{r.run}\n{r.season} d{int(r.day)}" for _,r in q.iterrows()];vals=100*q.relative_gap.to_numpy(float);bars=ax.bar(np.arange(len(q)),vals,color=['#E69F00' if v<=1.00005 else '#CC79A7' for v in vals]);ax.axhline(.5,ls='--',color='#555',label='Original 0.5% target');ax.axhline(1.0,ls=':',color='#333',label='1% reporting reference');ax.set_xticks(np.arange(len(q)),labels,rotation=45,ha='right');ax.set_ylabel('Native relative MIP gap (%)');ax.set_title('Time-limit incumbents retained without inventing the missing schedules');ax.legend(frameon=False);finish(fig,'TimeLimit_Incumbent_Gaps','These incumbents support objective/bound/gap reporting only; missing day-7 schedules are not reconstructed.')
    spfile=out/'SP_3Hz_hourly_worst.csv'
    if spfile.is_file():
        d=pd.read_csv(spfile);fig,ax=plt.subplots(figsize=(8.5,4.3));ax.plot(d.hour,d.worst_observed_hz,'o-',color='#7B5AA6');ax.axhline(FMIN,ls='--',color='#444',label='47 Hz (3-Hz drop)');ax.set(xlabel='Hour of source spring day 2',ylabel='Worst observed frequency (Hz)',title='Source-derived SP schedule: one-day 3-Hz post-UC assessment');ax.legend(frameon=False);finish(fig,'SP_3Hz_OneDay','Dedicated one-day 3-Hz comparator. It is not a four-season SP result and is not pooled with the historic 3.5-Hz weekly study.')
    atomic_csv(pd.DataFrame(register),figs/'Figure_Register.csv')
    return register

def collect_return(dep,out):
    target=EVID/('RETURN_'+dep['campaign']+'.zip');tmp=target.with_suffix('.tmp');files={}
    roots=[GAMS/'Results_Seasonal'/dep['campaign'],WORK,NB,PROJECT/'Documentation',GAMS/'inc']
    for root in roots:
        for p in root.rglob('*'):
            if p.is_file() and p.suffix.lower() not in ('.gdx','.png','.zip','.tmp','.lxi') and '.ipynb_checkpoints' not in p.parts:
                files[p.relative_to(PROJECT).as_posix()]=p
    for p in [GAMS/'LaPalma_FCUC_Weekly.gms',GAMS/'Input_Data_LaPalma_Weekly.gms',GAMS/'Run_LaPalma_Seasons.gms',GAMS/'LaPalma_Deployment.json',DATA/'LP_Case.json',PROJECT/'RUN_LAPALMA.md']:files[p.relative_to(PROJECT).as_posix()]=p
    manifest={key:dict(bytes=p.stat().st_size,sha256=sha_file(p)) for key,p in files.items()}
    # Binary native states stay locally; identify them without forcing a huge upload.
    for p in (GAMS/'Results_Seasonal'/dep['campaign']).rglob('*.gdx'):manifest[p.relative_to(PROJECT).as_posix()]=dict(bytes=p.stat().st_size,sha256=sha_file(p),included=False)
    with ZipFile(tmp,'w',ZIP_DEFLATED,compresslevel=6) as z:
        for key,p in files.items():z.write(p,key)
        z.writestr('RETURN_MANIFEST.json',json.dumps(manifest,indent=2))
    os.replace(tmp,target);print('RETURN package:',target,'; save executed Notebook04 and attach it separately.');return target


### Source code stage 6

**Algorithm objects:** `plot_learning`, `plot_matched_trajectories`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

def plot_learning(info):
    """Only frozen tables/predictions are plotted. No model fitting or ODE calls."""
    figs=LEARN/'Figures';figs.mkdir(exist_ok=True);register=[]
    table=pd.read_csv(LEARN/'Metrics.csv');pred=pd.read_csv(LEARN/'Test_predictions.csv');bins=pd.read_csv(LEARN/'Reliability_bins.csv')
    palette=['#0072B2','#56B4E9','#E69F00','#D55E00','#CC79A7','#009E73','#735F99','#A97B43','#267B78']
    def save(fig,name,caption):
        fig.tight_layout();fig.savefig(figs/(name+'.pdf'),bbox_inches='tight');fig.savefig(figs/(name+'.png'),dpi=600,bbox_inches='tight');plt.close(fig);register.append(dict(figure=name,caption=caption))
    for model in BUILDERS:
        candidate='cheap__core4__'+model
        for q in [.50,.99]:
            r=table[(table.candidate==candidate)&(table.partition=='test')&(table.pstar==q)].iloc[0]
            vals=np.array([[r.TP,r.FN],[r.FP,r.TN]],int);den=vals.sum(1,keepdims=True);pct=np.divide(vals,den,out=np.zeros_like(vals,dtype=float),where=den>0)
            fig,ax=plt.subplots(figsize=(5.2,4.2));im=ax.imshow(pct,cmap='Blues',vmin=0,vmax=1)
            for i in range(2):
                for j in range(2):ax.text(j,i,f'{vals[i,j]:,}\n{100*pct[i,j]:.2f}%',ha='center',va='center',color='white' if pct[i,j]>.55 else '#192333')
            ax.set_xticks([0,1],['Reject / unsafe','Accept / safe']);ax.set_yticks([0,1],['Actually unsafe','Actually safe']);ax.set_title(f'{model}; p*={q:.2f}; untouched test groups');fig.colorbar(im,ax=ax,label='Fraction within actual-class row');save(fig,f'CM_{model}_{q:.2f}','Raw counts plus row-normalized fractions. FN means an unsafe point accepted by the rule.')
    fig,ax=plt.subplots(figsize=(8,5))
    for k,model in enumerate(BUILDERS):
        d=pred[pred.candidate=='cheap__core4__'+model];pr,re,_=precision_recall_curve(d.y_safe==-1,1-d.p_safe);ax.plot(re,pr,color=palette[k],label=model)
    prevalence=float((d.y_safe==-1).mean());ax.axhline(prevalence,ls='--',color='#555555',label='Test unsafe fraction');ax.set(xlabel='Unsafe recall',ylabel='Unsafe precision',title='All nine learners; common held-out population');ax.legend(fontsize=8,ncol=2);save(fig,'Unsafe_PR','Precision-recall curves use unsafe as the event class; baseline is the test prevalence.')
    fig,ax=plt.subplots(figsize=(7.5,5));ax.plot([0,1],[0,1],':',color='#555555')
    for k,model in enumerate(BUILDERS):
        b=bins[bins.candidate=='cheap__core4__'+model];ax.plot(b.mean_probability_safe,b.observed_safe_fraction,'o-',markersize=3,color=palette[k],label=model)
    ax.set(xlabel='Mean predicted safe probability',ylabel='Observed safe fraction',title='Reliability: populated bins only');ax.legend(ncol=2,fontsize=8);save(fig,'Reliability','Bin counts and Wilson intervals are in the CSV. This is calibration against model-based labels, not physical risk calibration.')
    fig,ax=plt.subplots(figsize=(8.5,4.5))
    for k,model in enumerate(['LR_C1','PA']):
        d=table[(table.candidate=='cheap__core4__'+model)&(table.partition=='test')].sort_values('pstar');ax.plot(d.pstar,d.FNR,'o-',color=palette[k],label=model+' missed unsafe');ax.plot(d.pstar,d.FPR,'s--',color=palette[k],label=model+' rejected safe')
    ax.set(xlabel='Safe-probability floor',ylabel='Class-conditional error fraction',title='Threshold tradeoff before optimization');ax.legend(fontsize=8);save(fig,'Threshold_errors','The probability floor shifts the same calibrated score; it is not a guaranteed physical failure probability.')
    d=pred[pred.candidate=='cheap__core4__LR_C1'];fig,ax=plt.subplots(figsize=(8,4.5))
    score=np.log(np.clip(d.p_safe,1e-12,1-1e-12)/(1-np.clip(d.p_safe,1e-12,1-1e-12)))
    for label,col in [(1,'#0072B2'),(-1,'#D55E00')]:
        mask=d.y_safe==label;ax.scatter(score[mask],d.loc[mask,'minimum_observed_hz'],s=8,alpha=.5,color=col,label='Safe label' if label==1 else 'Unsafe label')
    ax.axhline(FMIN,ls='--',color='#333333');ax.axvline(math.log(99),ls=':',color='#333333');ax.set(xlabel='Calibrated logit',ylabel='Observed model minimum (Hz)',title='LR-C1 score and model outcome; not field ground truth');ax.legend();save(fig,'Score_vs_minimum','Model-based minima; a censored value, where present, is a stopping observation rather than an eventual nadir.')
    atomic_csv(pd.DataFrame(register),figs/'Figure_Register.csv');return register


def plot_matched_trajectories(dep,out,tables):
    """One common point per season: day1/hour1, largest-producing unit in BC.
    The SAME online unit must exist in every plotted method; missing events are
    disclosed rather than silently replacing the outage by another generator.
    """
    out=Path(out);figs=out/'Figures';figs.mkdir(exist_ok=True);records=[]
    for season in CASE['seasons']:
        base=GAMS/'Results_Seasonal'/dep['campaign']/'BC'/season/'d1'/f'Results_BC_{season}_d1.xlsx'
        if not base.is_file():continue
        rec=load_daily(base,'BC',season,1,dep);j=int(np.argmax(rec['power'][0]));curves=[]
        for run in ['BC','RESERVE','SP','LR_C1','PA']:
            book=GAMS/'Results_Seasonal'/dep['campaign']/run/season/'d1'/f'Results_{run}_{season}_d1.xlsx'
            if not book.is_file():continue
            rec=load_daily(book,run,season,1,dep)
            if rec['commitment'][0,j]<.5:
                records.append(dict(season=season,run=run,outage=IDS[j],status='NOT_COMPARABLE_OUTAGE_UNIT_OFF'));continue
            check,tr=integrate_event(rec['power'][0],rec['demand'][0],j,rec['commitment'][0],refined=True,return_trace=True)
            hit=tables['All_events'];hit=hit[(hit.season==season)&(hit.run==run)&(hit.global_hour==1)&(hit.outage_unit==IDS[j])]
            if len(hit)!=1 or not np.isfinite(float(hit.iloc[0].minimum_observed_hz)) or abs(float(hit.iloc[0].minimum_observed_hz)-check['minimum_observed_hz'])>.001:raise ValueError('Trace replay does not match a finite stored event')
            records.append(dict(season=season,run=run,outage=IDS[j],status='REPLAY_MATCHED',loss_mw=check['loss_mw'],min_hz=check['minimum_observed_hz']))
            atomic_csv(pd.DataFrame(tr),figs/f'Trace_{season}_{run}_{IDS[j]}.csv');curves.append((run,tr,check['loss_mw']))
        for var,ylabel,name in [('frequency_hz','Aggregate frequency (Hz)','Frequency'),('response_mw','Delivered incremental response (MW)','Response')]:
            fig,ax=plt.subplots(figsize=(8.5,4.5))
            for k,(run,tr,loss) in enumerate(curves):mask=tr['time_s']<=20;ax.plot(tr['time_s'][mask],tr[var][mask],label=f'{run}; loss {loss:.2f} MW',color=['#4B5563','#009E73','#735F99','#0072B2','#D55E00'][k])
            if var=='frequency_hz':ax.axhline(FMIN,color='#555555',ls='--')
            ax.set(xlabel='Seconds after the same generator trips',ylabel=ylabel,title=season.title()+f' day1/hour1: outage {IDS[j]} (actual scheduled output)');ax.legend(fontsize=8);fig.tight_layout();fig.savefig(figs/f'{name}_{season}.pdf');fig.savefig(figs/f'{name}_{season}.png',dpi=600);plt.close(fig)
    atomic_csv(pd.DataFrame(records),figs/'Trace_Comparability.csv');return records


## Load the exact native campaign and check source identities

### Source code stage 8

**Algorithm objects:** `_normalize_reserved_label_patch`.

These definitions implement the original routines; inspect their input validation and return fields before calling them. Defining a function does not itself verify its numerical behavior.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
deployment=json.loads((GAMS/'LaPalma_Deployment.json').read_text())
assert deployment['model_id']==MODEL_ID, 'Different dynamic definition; do not mix labels and schedules.'

# Accept the documented syntax-only GAMS 24.9 repair that quotes the reserved
# set label model. All other frozen sources must still match exactly.
def _normalize_reserved_label_patch(text):
    return text.replace("set idk /campaign,run,season,'model'/;",
                        "set idk /campaign,run,season,model/;")

source_snapshot = GAMS/'Results_Seasonal'/deployment['campaign']/'source_snapshot'
for relative,expected in deployment['sources'].items():
    live = PROJECT/relative
    actual = sha_file(live)
    if actual == expected:
        continue
    if relative == 'GAMS_LaPalma/LaPalma_FCUC_Weekly.gms':
        snap = source_snapshot/'LaPalma_FCUC_Weekly.gms'
        if not snap.is_file() or sha_file(snap) != expected:
            raise ValueError('Frozen source snapshot is missing or inconsistent for '+relative)
        live_text = live.read_text(encoding='utf-8')
        snap_text = snap.read_text(encoding='utf-8')
        if _normalize_reserved_label_patch(live_text) == _normalize_reserved_label_patch(snap_text):
            print('Accepted documented GAMS 24.9 syntax-only repair for', relative)
            continue
    raise ValueError('Frozen native input/source changed beyond the approved syntax repair: '+relative)

if sha_file(GAMS/'inc/LaPalma_Calibrated.inc')!=deployment['cut_sha256']:
    raise ValueError('Deployed coefficients changed')

# Preflight the native campaign BEFORE any expensive post-UC ODE evaluation.
# SP is intentionally separated from the four-season production denominator:
# the paper retains the historical 3.5-Hz analytical comparator and the current
# dedicated 3-Hz source day; a four-season 3-Hz SP rerun is not required here.
root = GAMS/'Results_Seasonal'/deployment['campaign']
primary_modes=[m for m in deployment['runs'] if m['run']!='SP']
complete=[]; failed=[]; blocked=[]
for mode in primary_modes:
    run=mode['run']
    for season in CASE['seasons']:
        predecessor_ok=True
        for day in range(1,8):
            folder=root/run/season/f'd{day}'
            if (folder/'COMPLETE.txt').is_file():
                complete.append((run,season,day)); predecessor_ok=True
            elif (folder/'FAILED_SOLVE.txt').is_file():
                failed.append((run,season,day)); predecessor_ok=False
            else:
                blocked.append((run,season,day,'BLOCKED_PREDECESSOR' if not predecessor_ok else 'MISSING')); predecessor_ok=False
sp_attempts=[]
for season in CASE['seasons']:
    folder=root/'SP'/season/'d1'
    if (folder/'FAILED_SOLVE.txt').is_file(): sp_attempts.append(('SP',season,1))
print('Reading frozen seasonal campaign',deployment['campaign'])
print(f'Primary four-season production scope: {len(primary_modes)*4*7} non-SP daily jobs.')
print(f'Native preflight: {len(complete)} strict COMPLETE, {len(failed)} time-limit/failed day(s), {len(blocked)} blocked/missing.')
print('Post-hoc reporting policy: original solver target remains 0.5%; time-limit incumbents are retained descriptively with their exact gaps. No missing schedule is invented.')
if failed:
    print('Primary time-limit/failed days (Notebook04 will parse incumbent objective/bound/gap where available):')
    for item in failed: print('  ',item)
print('SP scope: current source-derived spring d2 workbook at 3-Hz criterion is processed as ONE DAY only; historic Raja/submitted SP at 3.5 Hz remains contextual evidence.')
print('Exploratory native SP day-1 attempts found:',sp_attempts)


## Schedule checks and cached post-contingency evaluation
This is the only potentially long cell. Each accepted hourly operating point is evaluated for every producing online unit. The event identity includes the actual dispatch, load, commitment, outage, model and numerical settings. Records are cached and written before Excel. An i11 commitment is an explicit training-domain flag, not automatically an error or a reason to remove the unit.

### Source code stage 10

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
OUTROOT,tables=process_seasons(deployment)
display(tables['Seasonal_summary'])
print('\nTime-limit incumbents retained descriptively (exact gaps):')
display(tables['Relaxed_incumbents'])
print('\nDedicated 3-Hz SP one-day comparator:')
display(tables['SP_3Hz_one_day_summary'])
print('All CSV checkpoints are saved before Excel. Do not repeat GAMS or SODE calculations merely to repair reporting files.')


## Output-only Excel cell
Repeat this cell alone after a spreadsheet error. The writer never deletes a locked prior workbook.

### Source code stage 12

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
workbook=write_excel(OUTROOT/'LP_Seasonal_Comparison.xlsx',tables)
print('Workbook:',workbook)

## Publication figures
Histograms/KDEs include only settled, uncensored, numerically accepted minima. Outcome accounting preserves exclusions. The selected trajectory pair uses the same generator at the same day/hour; its actual output/loss may differ by method and is stated on the plot. No analytical-SP time curve is invented.

### Source code stage 14

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
figure_register=publication_figures(OUTROOT,tables)
trace_checks=plot_matched_trajectories(deployment,OUTROOT,tables)
print('Figures saved as PDF and 600-dpi PNG.')

## Lightweight return package
The archive contains the numerical tables, native Excel/listings, labels, models, source and PDF figures. GDX and large PNGs remain locally and are listed by hash. Save this executed notebook after the final cell and attach it separately with the RETURN ZIP.

### Source code stage 16

This cell uses objects created in preceding cells. Check the immediately preceding source explanation and the declared input files before execution.

**Verification:** original code-cell text is unchanged and syntax-checked. Printed values and figures become new evidence only after their required inputs have been supplied and execution actually succeeds. Preserve caches/receipts and distinguish a failed calculation from an unsafe physical label.

In [ ]:
return_zip=collect_return(deployment,OUTROOT)